# Sentinel-1 L0 → SLC Batch Order and S3 Upload

Submits [Batch Orders](https://documentation.dataspace.copernicus.eu/APIs/On-Demand%20Production%20API.html)
for a list of Sentinel-1 L0 scenes via the CDSE On-Demand Production API. For our international partner account, 
we are limited to 10 scenes at a time and one active batch at a time. This code breaks a scene list into batches of
10, submits them, and then then downloads, unzips, and uploads each completed SLC product to S3. See
[`sentinel1_l0_to_slc_ondemand.ipynb`](./sentinel1_l0_to_slc_ondemand.ipynb) for the single-scene
walkthrough, and [`search_ew_l0_scenes.ipynb`](./search_ew_l0_scenes.ipynb) to build
`L0_PRODUCTS` from an AOI.

**Steps:** define functions → authenticate (CDSE + AWS) → check L0 availability (LTA) →
process all scenes in batches of 10, submitting/resuming/polling/uploading each batch in turn.
Safe to interrupt and re-run from the top -- already-submitted batches and uploaded scenes are
never resubmitted or re-uploaded.

## Configuration

Set the L0 product list and S3 destination. Credentials come from
`CDSE_USERNAME`/`CDSE_PASSWORD` env vars (or enter directly).

In [ ]:
import os
import json
from datetime import date
from dotenv import load_dotenv

try:
    # load credentials from root .env file
    load_dotenv("../../.env")
except:
    print("Could not find .env file with credentials.")

# --- INPUT -------------------------------------------------------------------
# List of L0 products to process, 10 at a time, as a series of Batch Orders.
# See search_ew_l0_scenes.ipynb to build this list from an AOI, e.g. via:
# L0_PRODUCTS = json.load(open("EW_L0_scene_list.txt"))
# L0_PRODUCTS = [line.strip() for line in open("EW_L0_scene_list.txt") if line.strip()]
L0_PRODUCTS = [
    "S1A_EW_RAW__0SSH_20220131T034822_20220131T034930_041699_04F623_CCA2.SAFE",
]

# A label used to build deterministic, resumable sub-batch names (<NAME>_chunk0000,
# _chunk0001, ...). Keep this unchanged across restarts of section 3's batch loop so it
# can find already-submitted chunks instead of resubmitting them.
BATCH_ORDER_NAME = f"slc_batch_{len(L0_PRODUCTS)}_scenes"

# CDSE limit: max 10 scenes per BatchOrder
BATCH_CHUNK_SIZE = 10

# Wait before re-checking AWS credentials after a failure, in section 3's upload step
CREDENTIAL_RETRY_SECONDS = 300

# Where to save downloaded/unzipped files locally before upload
OUTPUT_DIR = "."

# Scenes whose batch item ultimately failed are appended here as section 3 runs, so
# progress survives a kernel/instance restart instead of only living in memory.
FAILED_SCENES_FILE = os.path.join(OUTPUT_DIR, "failed_scenes.txt")

# Number of scenes to download/unzip/upload concurrently (per batch) in section 3. Reduce
# this if you hit rate limits or connection errors against CDSE object storage or S3.
MAX_WORKERS = 2

# S3 destination for the unzipped SLC products produced by the batch
S3_BUCKET = "deant-data-public-dev"
S3_PROJECT_FOLDER = "s1_ew_slc"
S3_SCENE_UPLOAD_FOLDER = f"{S3_PROJECT_FOLDER}/data"
S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER = f"{S3_PROJECT_FOLDER}/tracking/L0_RAW"
S3_L1_SLC_SCENE_TRACKING_UPLOAD_FOLDER = f"{S3_PROJECT_FOLDER}/tracking/L1_SLC"

# AWS profile to use for S3 uploads. Set this directly rather than relying
# On the default picked up on the credential refresh
# An AWS projile can also be supplied in the .env file
AWS_PROFILE_OVERRIDE = ""  # e.g. "SuperDeveloper"

# Credentials — prefer env vars so they are not committed
CDSE_USERNAME = os.environ.get("CDSE_LOGIN", "")  # or set directly: "your@email.com"
CDSE_PASSWORD = os.environ.get("CDSE_PASSWORD", "")  # or set directly: "yourpassword"
# -----------------------------------------------------------------------------

WORKFLOW_NAME = "Sentinel-1-L0-EW_SLC__1S"
BASE_URL = "https://odp.dataspace.copernicus.eu/odata/v1"
CATALOGUE_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1"
ZIPPER_URL = "https://zipper.dataspace.copernicus.eu/odata/v1"
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

print(f"Scenes     : {len(L0_PRODUCTS)}")
print(f"Batch name : {BATCH_ORDER_NAME}")
print(f"Output dir : {os.path.abspath(OUTPUT_DIR)}")
print(f"S3 target  : s3://{S3_BUCKET}/{S3_SCENE_UPLOAD_FOLDER}")

## Functions

Every reusable function, class, and shared constant used by the rest of this notebook lives
here, grouped by purpose. The numbered sections below are just drivers that call into these —
run this whole section once (after Configuration) before running anything else.

In [ ]:
import json
import time
import getpass
import zipfile
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import boto3
from botocore.exceptions import ClientError, NoCredentialsError, TokenRetrievalError

### CDSE Authentication

In [ ]:
def get_token(username: str, password: str) -> str:
    resp = requests.post(
        TOKEN_URL,
        data={
            "client_id": "cdse-public",
            "username": username,
            "password": password,
            "grant_type": "password",
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    if not resp.ok:
        try:
            detail = resp.json()
        except Exception:
            detail = resp.text
        raise RuntimeError(
            f"Authentication failed ({resp.status_code}): {detail}\n\n"
            "Check that:\n"
            "  1. Your Copernicus Data Space account is verified at https://dataspace.copernicus.eu\n"
            "  2. You are using your registration email + password (not a Google/GitHub SSO login)\n"
            "  3. There are no leading/trailing spaces in your credentials"
        )
    data = resp.json()
    return data["access_token"], data.get("expires_in", 600)


def auth_headers() -> dict:
    """Return headers with a fresh token, refreshing if within 60s of expiry."""
    global TOKEN, TOKEN_ACQUIRED_AT, TOKEN_EXPIRES_IN
    if time.monotonic() - TOKEN_ACQUIRED_AT > (TOKEN_EXPIRES_IN - 60):
        print("Refreshing token...")
        TOKEN, TOKEN_EXPIRES_IN = get_token(CDSE_USERNAME, CDSE_PASSWORD)
        TOKEN_ACQUIRED_AT = time.monotonic()
    return {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

### L0 Availability (LTA)

In [ ]:
def get_product_by_name(name: str) -> dict:
    resp = requests.get(
        f"{CATALOGUE_URL}/Products",
        params={
            "$filter": f"Name eq '{name}'",
            "$select": "Id,Name,Online,ContentLength",
        },
        headers=auth_headers(),
        timeout=30,
    )
    resp.raise_for_status()
    products = resp.json().get("value", [])
    if not products:
        raise RuntimeError(
            f"Product '{name}' not found in the CDSE catalogue — it may have been purged "
            "from the archive entirely rather than simply being offline in the LTA."
        )
    return products[0]


def trigger_lta_retrieval(product_id: str) -> int:
    """GET the product's $value URL to trigger an LTA restore; connection closes right
    after headers arrive, before the (potentially multi-GB) body streams."""
    with requests.get(
        f"{ZIPPER_URL}/Products({product_id})/$value",
        headers=auth_headers(),
        stream=True,
        timeout=60,
    ) as r:
        return r.status_code


def wait_for_online(
    product_id: str, poll_interval_seconds: int = 300, timeout_hours: float = 24.0
) -> None:
    deadline = time.monotonic() + timeout_hours * 3600
    while True:
        resp = requests.get(
            f"{CATALOGUE_URL}/Products({product_id})",
            params={"$select": "Online"},
            headers=auth_headers(),
            timeout=30,
        )
        resp.raise_for_status()
        online = resp.json()["Online"]
        print(f"[{datetime.utcnow().strftime('%H:%M:%S')} UTC]  Online: {online}")
        if online:
            return
        if time.monotonic() > deadline:
            raise TimeoutError(
                f"Product {product_id} did not come online within {timeout_hours}h "
                "of triggering LTA retrieval."
            )
        time.sleep(poll_interval_seconds)

### AWS S3 Credentials

In [ ]:
def get_checked_s3_client() -> boto3.client:
    """Build a fresh boto3 S3 client, confirm the resolved credentials are valid, and
    confirm they can write to S3_BUCKET. Raises with actionable guidance instead of the
    raw botocore traceback if credentials are missing or an SSO token has expired."""
    profile = AWS_PROFILE_OVERRIDE or os.environ.get("AWS_PROFILE")
    if profile:
        print(f"Using AWS profile '{profile}' -- this takes priority over an EC2 instance role.")

    session = boto3.Session(profile_name=profile) if profile else boto3.Session()

    try:
        identity = session.client("sts").get_caller_identity()
        print(f"AWS credentials OK: {identity['Arn']}")
    except TokenRetrievalError:
        raise RuntimeError(
            f"SSO token for profile '{profile}' has expired and could not refresh "
            "automatically (refresh requires an interactive browser login). Run on the "
            f"instance:\n\n    aws sso login --profile {profile}\n\nthen re-run this cell. "
            "If this EC2 instance has an IAM role attached, consider clearing "
            "AWS_PROFILE_OVERRIDE and AWS_PROFILE (and any AWS_ACCESS_KEY_ID/"
            "AWS_SECRET_ACCESS_KEY/AWS_SESSION_TOKEN env vars) so boto3 falls back to the "
            "instance role instead, which refreshes on its own."
        )
    except NoCredentialsError:
        raise RuntimeError(
            "No AWS credentials found."
        )

    client = session.client("s3")
    test_key = f"{S3_SCENE_UPLOAD_FOLDER.rstrip('/')}/.permission_check_{os.getpid()}"
    try:
        client.put_object(Bucket=S3_BUCKET, Key=test_key, Body=b"permission check")
        client.delete_object(Bucket=S3_BUCKET, Key=test_key)
        print(f"Upload permission OK for s3://{S3_BUCKET}/{S3_SCENE_UPLOAD_FOLDER}")
    except ClientError as e:
        print(f"Upload permission check FAILED: {e.response['Error']['Code']} — {e.response['Error']['Message']}")
        raise

    return client

### Batch Orders (CDSE On-Demand Production API)

In [ ]:
NON_REUSABLE_STATUSES = {"failed", "cancelled"}
TERMINAL_STATUSES = {"completed", "failed", "cancelled"}
POLL_INTERVAL_SECONDS = 60


def get_batch_order_by_name(name: str) -> dict | None:
    """Look up a previously-submitted BatchOrder by Name; None if none exists."""
    resp = requests.get(
        f"{BASE_URL}/BatchOrder",
        params={"$filter": f"Name eq '{name}'"},
        headers=auth_headers(),
        timeout=30,
    )
    resp.raise_for_status()
    orders = resp.json().get("value", [])
    if not orders:
        return None
    orders.sort(key=lambda o: o.get("SubmissionDate", ""), reverse=True)
    return orders[0]


def submit_batch_order(name: str, l0_products: list[str]) -> dict:
    batch_payload = {
        "Name": name,
        "WorkflowName": WORKFLOW_NAME,
        "IdentifierList": l0_products,
        "WorkflowOptions": [],
        "Priority": 1,
    }
    resp = requests.post(
        f"{BASE_URL}/BatchOrder/OData.CSC.Order",
        headers=auth_headers(),
        json=batch_payload,
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json().get("value", resp.json())


def get_batch_order_status(batch_order_id: str) -> dict:
    r = requests.get(
        f"{BASE_URL}/BatchOrder({batch_order_id})",
        headers=auth_headers(),
        timeout=30,
    )
    if not r.ok:
        print(f"  Warning: {r.status_code} — {r.text[:200]}")
        r.raise_for_status()
    data = r.json()
    return data.get("value", data)


def list_batch_order_items(batch_order_id: str) -> list[dict]:
    resp = requests.get(
        f"{BASE_URL}/BatchOrder({batch_order_id})/Products",
        headers=auth_headers(),
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    items = data.get("value", data)
    if not isinstance(items, list):
        items = [items]
    return items

### Download, Unzip, and Upload to S3

In [ ]:
_print_lock = threading.Lock()


def log(message: str) -> None:
    """Thread-safe print so parallel workers don't interleave mid-line."""
    with _print_lock:
        print(message)


class _HTTPRangeFile:
    """Read+seek file-like object backed by HTTP Range requests, so zipfile can read just
    the central directory of a remote zip without downloading the body."""

    def __init__(self, url: str, size: int):
        self._url = url
        self._size = size
        self._pos = 0

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self._pos

    def seek(self, offset: int, whence: int = 0) -> int:
        if whence == 0:
            self._pos = offset
        elif whence == 1:
            self._pos += offset
        elif whence == 2:
            self._pos = self._size + offset
        else:
            raise ValueError(f"Unsupported whence: {whence}")
        return self._pos

    def read(self, size: int = -1) -> bytes:
        if size is None or size < 0:
            end = self._size - 1
        else:
            end = min(self._pos + size, self._size) - 1
        if self._pos > end:
            return b""
        resp = requests.get(self._url, headers={"Range": f"bytes={self._pos}-{end}"}, timeout=60)
        if resp.status_code not in (200, 206):
            resp.raise_for_status()
        data = resp.content
        self._pos += len(data)
        return data


def peek_remote_inner_zip_name(download_link: str, known_size: int | None = None) -> str:
    """Return the inner filename inside a remote order .zip via Range requests against its
    central directory -- no download, no unzip. Raises if ranges aren't supported or the
    archive doesn't hold exactly one member, so the caller can fall back to download+unzip."""
    probe = requests.get(download_link, headers={"Range": "bytes=0-0"}, timeout=30)
    if probe.status_code != 206:
        raise RuntimeError(
            f"Server did not honor a Range request (status {probe.status_code}); "
            "cannot peek without downloading."
        )
    content_range = probe.headers.get("Content-Range", "")
    try:
        total_size = int(content_range.rsplit("/", 1)[-1])
    except (ValueError, IndexError):
        if not known_size:
            raise RuntimeError(f"Could not parse total size from Content-Range: {content_range!r}")
        total_size = known_size

    remote_file = _HTTPRangeFile(download_link, total_size)
    with zipfile.ZipFile(remote_file) as zf:
        names = zf.namelist()
    inner_zips = [n for n in names if n.lower().endswith(".zip")]
    if len(inner_zips) != 1:
        raise RuntimeError(f"Expected exactly one inner .zip, found: {names}")
    return inner_zips[0]


def get_local_inner_zip_name(zip_path: str) -> str:
    """Return the single inner .zip member name inside a local order .zip (no extraction)."""
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
    inner_zips = [n for n in names if n.lower().endswith(".zip")]
    if len(inner_zips) != 1:
        raise RuntimeError(f"Expected exactly one inner .zip in {zip_path}, found: {names}")
    return inner_zips[0]


def download_batch_item(download_link: str, local_path: str) -> None:
    """Download a batch item's product from its pre-signed Swift `DownloadLink`."""
    with requests.get(download_link, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)


def build_tracking_record(
    item: dict,
    l1_product: str,
    s3_bucket: str,
    s3_scene_key: str,
    batch_order_id,
    batch_order_name: str,
) -> dict:
    """Tracking JSON for one uploaded scene. Batch items carry no `WorkflowOptions`, so
    processor_platform/version are always null here. batch_order_id/batch_order_name are
    passed in explicitly (rather than read from globals) so this is correct for whichever
    chunk's batch order the caller is processing."""
    workflow_options = {
        o.get("Name"): o.get("Value") for o in item.get("WorkflowOptions", []) or []
    }
    return {
        "l0_product": item.get("InputProductReference"),
        "l1_product": l1_product,
        "workflow_name": item.get("WorkflowName", WORKFLOW_NAME),
        "processor_platform": workflow_options.get("platform"),
        "processor_version": workflow_options.get("version"),
        "source": "CDSE On-Demand Production API (BatchOrder)",
        "batch_order_id": batch_order_id,
        "batch_order_name": batch_order_name,
        "order_item_id": item.get("Id"),
        "order_item_status": item.get("Status"),
        "order_submission_date": item.get("SubmissionDate"),
        "date_created": datetime.utcnow().isoformat() + "Z",
        "s3_bucket": s3_bucket,
        "s3_scene_key": s3_scene_key,
    }


def key_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "404":
            return False
        raise  # re-raise anything that isn't "not found" (e.g. permissions)


def process_batch_item(item: dict, batch_order_id, batch_order_name: str) -> str:
    """Download a batch item's order package. The package is a zip file, containing the
    zipped SAFE file (i.e. a nested zip -> BATCH_ID.zip/SCENE.zip/SCENE.SAFE). Rather than unzipping
    and uploading the SCENE.ZIP, we instead rename the BATCH_ID.zip to share the same name as
    the scene zip. The result is a nested zip, which has a smaller storage size. This must be
    handled downstream.  
    """
    item_id = item.get("Id")
    item_status = str(item.get("Status", "")).lower()
    l0_reference = item.get("InputProductReference") or f"item_{item_id}"
    item_label = os.path.splitext(os.path.basename(str(l0_reference)))[0]

    if item_id is None:
        log(f"Skipping item with no Id field: {item}")
        return "skipped"
    if item_status and item_status != "completed":
        log(f"Skipping {item_label} (Id={item_id}): status={item_status}")
        return "skipped"

    download_link = item.get("DownloadLink")
    if not download_link:
        log(f"Skipping {item_label} (Id={item_id}): no DownloadLink present.")
        return "skipped"

    log(f"[{item_label}] Processing (Id={item_id})...")

    l1_product_name = None
    try:
        l1_product_name = os.path.basename(peek_remote_inner_zip_name(download_link, item.get("ProcessedSize")))
    except Exception as e:
        log(f"[{item_label}] Could not peek remote zip contents ({e}); will determine name after download.")

    if l1_product_name is not None:
        if not l1_product_name.endswith(".zip"):
            l1_product_name += ".zip"
        s3_key = f"{S3_SCENE_UPLOAD_FOLDER.rstrip('/')}/{l1_product_name}"
        if key_exists(s3_client, S3_BUCKET, s3_key):
            log(f"[{item_label}] Already uploaded -> s3://{S3_BUCKET}/{s3_key} (skipping download/upload)")
            return "already_uploaded"

    order_zip_path = os.path.join(OUTPUT_DIR, f"{item_label}.zip")
    if not os.path.exists(order_zip_path):
        log(f"[{item_label}] Downloading order package -> {order_zip_path}")
        download_batch_item(download_link, order_zip_path)
    else:
        log(f"[{item_label}] Order package exists, download skipped -> {order_zip_path}")

    if l1_product_name is None:
        l1_product_name = os.path.basename(get_local_inner_zip_name(order_zip_path))
        if not l1_product_name.endswith(".zip"):
            l1_product_name += ".zip"

    l1_local_path = os.path.join(OUTPUT_DIR, l1_product_name)
    if os.path.abspath(order_zip_path) != os.path.abspath(l1_local_path):
        log(f"[{item_label}] Renaming order package -> {l1_local_path}")
        os.replace(order_zip_path, l1_local_path)

    s3_key = f"{S3_SCENE_UPLOAD_FOLDER.rstrip('/')}/{l1_product_name}"
    if key_exists(s3_client, S3_BUCKET, s3_key):
        log(f"[{item_label}] Already uploaded -> s3://{S3_BUCKET}/{s3_key} (skipping upload)")
        return "already_uploaded"

    log(f"[{item_label}] Uploading -> s3://{S3_BUCKET}/{s3_key}")
    s3_client.upload_file(l1_local_path, S3_BUCKET, s3_key)
    log(f"[{item_label}] Successfully uploaded. Deleting file.")
    os.remove(l1_local_path)

    tracking_record = build_tracking_record(
        item, l1_product_name, S3_BUCKET, s3_key, batch_order_id, batch_order_name
    )
    # The same tracking JSON is uploaded to two folders under different names, so it can
    # be looked up later by either the L0 (input) or L1 (output) scene name.
    l0_tracking_filename = f"{item_label}.json"
    l1_tracking_filename = os.path.splitext(l1_product_name)[0] + ".json"
    tracking_local_path = os.path.join(OUTPUT_DIR, l1_tracking_filename)
    with open(tracking_local_path, "w") as f:
        json.dump(tracking_record, f, indent=2)

    l0_tracking_s3_key = f"{S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER.rstrip('/')}/{l0_tracking_filename}"
    s3_client.upload_file(tracking_local_path, S3_BUCKET, l0_tracking_s3_key)
    log(f"[{item_label}] Uploaded tracking record -> s3://{S3_BUCKET}/{l0_tracking_s3_key}")

    l1_tracking_s3_key = f"{S3_L1_SLC_SCENE_TRACKING_UPLOAD_FOLDER.rstrip('/')}/{l1_tracking_filename}"
    s3_client.upload_file(tracking_local_path, S3_BUCKET, l1_tracking_s3_key)
    log(f"[{item_label}] Uploaded tracking record -> s3://{S3_BUCKET}/{l1_tracking_s3_key}")

    os.remove(tracking_local_path)
    return "uploaded"

### Batch Processing Loop (10 Scenes at a Time)

In [ ]:
# Scenes already recorded as failed, loaded from FAILED_SCENES_FILE if it exists, so
# record_failed_scene() stays deduplicated across kernel/instance restarts, not just
# within this session.
if os.path.exists(FAILED_SCENES_FILE):
    with open(FAILED_SCENES_FILE) as f:
        _failed_scenes_written = {line.strip() for line in f if line.strip()}
else:
    _failed_scenes_written = set()


def record_failed_scene(scene: str) -> None:
    """Append a failed scene to FAILED_SCENES_FILE immediately (deduplicated against both
    this session and any previous run's file), so failures are visible on disk as the loop
    progresses rather than only printed once at the very end."""
    if scene in _failed_scenes_written:
        return
    _failed_scenes_written.add(scene)
    with _print_lock:
        with open(FAILED_SCENES_FILE, "a") as f:
            f.write(scene + "\n")


def chunk_scenes(scenes: list[str], size: int) -> list[list[str]]:
    return [scenes[i : i + size] for i in range(0, len(scenes), size)]


def wait_for_valid_s3_client(retry_seconds: int = CREDENTIAL_RETRY_SECONDS) -> None:
    """Refresh/validate the global s3_client before uploading. Retries indefinitely on
    failure so a long-running loop can ride out a credential problem that gets fixed
    externally (e.g. an instance-role policy update, or someone re-running `aws sso
    login`) instead of crashing the whole run."""
    global s3_client
    while True:
        try:
            s3_client = get_checked_s3_client()
            return
        except Exception as e:
            print(f"AWS credential check failed: {e}\nRetrying in {retry_seconds}s...")
            time.sleep(retry_seconds)


def process_chunk(chunk_index: int, scenes: list[str]) -> dict[str, str]:
    """Submit (or resume) one <=10-scene batch, poll it to completion, then upload all
    completed items. Safe to re-run: batch names are deterministic and S3 upload is
    idempotent, so restarting the loop never resubmits or re-uploads completed work. Any
    item that ends up "failed" is recorded to FAILED_SCENES_FILE immediately."""
    chunk_name = f"{BATCH_ORDER_NAME}_chunk{chunk_index:04d}"

    order = get_batch_order_by_name(chunk_name)
    if order is None:
        print(f"\n[chunk {chunk_index}] Submitting new batch '{chunk_name}' ({len(scenes)} scenes)...")
        order = submit_batch_order(chunk_name, scenes)
    elif str(order.get("Status", "")).lower() in NON_REUSABLE_STATUSES:
        print(
            f"\n[chunk {chunk_index}] Existing batch '{chunk_name}' is "
            f"{order.get('Status')} -- submitting a new one instead of reusing it."
        )
        order = submit_batch_order(chunk_name, scenes)
    else:
        print(
            f"\n[chunk {chunk_index}] Resuming existing batch '{chunk_name}' "
            f"(Id={order['Id']}, Status={order.get('Status')})."
        )

    order_id = order["Id"]
    status = str(order.get("Status", "unknown")).lower()
    while status not in TERMINAL_STATUSES:
        print(f"[chunk {chunk_index}] [{datetime.utcnow().strftime('%H:%M:%S')} UTC]  Batch status: {status}")
        time.sleep(POLL_INTERVAL_SECONDS)
        order = get_batch_order_status(order_id)
        status = str(order.get("Status", "unknown")).lower()
    print(f"[chunk {chunk_index}] Batch finished with status: {status}")

    items = list_batch_order_items(order_id)

    wait_for_valid_s3_client()

    results = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_item = {
            executor.submit(process_batch_item, item, order_id, chunk_name): item
            for item in items
        }
        for future in as_completed(future_to_item):
            item = future_to_item[future]
            item_label = item.get("InputProductReference", item.get("Id"))
            try:
                results[item_label] = future.result()
            except Exception as e:
                log(f"[chunk {chunk_index}] [{item_label}] FAILED: {e}")
                results[item_label] = "failed"
            if results[item_label] == "failed":
                record_failed_scene(str(item_label))

    chunk_summary = {
        s: sum(1 for v in results.values() if v == s)
        for s in ("uploaded", "already_uploaded", "skipped", "failed")
    }
    print(f"[chunk {chunk_index}] Upload summary: {chunk_summary}")
    return results

## 1. Authenticate

In [ ]:
if not CDSE_USERNAME:
    CDSE_USERNAME = input("Copernicus username (email): ")
if not CDSE_PASSWORD:
    CDSE_PASSWORD = getpass.getpass("Copernicus password: ")

TOKEN, TOKEN_EXPIRES_IN = get_token(CDSE_USERNAME, CDSE_PASSWORD)
TOKEN_ACQUIRED_AT = time.monotonic()

print(f"Authenticated successfully. Token valid for {TOKEN_EXPIRES_IN}s.")

### AWS Credentials Check and Refresh

Validates that AWS credentials for S3 are resolvable and can actually write to `S3_BUCKET`
before the rest of the notebook runs. Run this again any time a later cell fails with
`TokenRetrievalError` or `NoCredentialsError`.

In [ ]:
s3_client = get_checked_s3_client()

## 2. Check L0 Input Product Availability (Online / LTA)

CDSE's order service is documented to pull offline inputs from LTA itself, so this is a
diagnostic step rather than a hard requirement. For each scene in `L0_PRODUCTS`: zero catalogue
results means it isn't in the archive at all (report to CDSE support); `Online: false` triggers a
restore attempt; `Online: true` needs nothing. Restores are triggered for all offline scenes up
front so the waits below run concurrently.

In [ ]:
l0_products_info = {}
for product_name in L0_PRODUCTS:
    info = get_product_by_name(product_name)
    l0_products_info[product_name] = info
    print(f"{product_name}: Online={info['Online']}")
    if not info["Online"]:
        print(f"  Triggering LTA retrieval for {product_name}...")
        trigger_lta_retrieval(info["Id"])

# Wait for any offline products to come back online (retrieval was triggered for all of
# them above, so the waits below overlap rather than compounding).
for product_name, info in l0_products_info.items():
    if not info["Online"]:
        print(f"\nWaiting for {product_name} to come online...")
        wait_for_online(info["Id"])
        print(f"{product_name} is now online.")

print("\nAll scenes in L0_PRODUCTS are online and ready to order.")

## 3. Process All Scenes in Batches of 10

Drops any scene already processed (found in `S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER`), then
submits (or resumes) a BatchOrder for each group of up to 10 remaining scenes in
`L0_PRODUCTS` -- CDSE's per-order limit, with only one active order at a time -- polls to
completion, then uploads to S3. Uses the functions defined in **Functions** above; only
Configuration, Functions, and section 1 (Authenticate) need to have run first. Section 2 is
optional.

Downloaded items are renamed (not unzipped) to the SAFE product's name before upload, so S3
holds a zip-of-a-zip -- `download_safe_file()` unwraps this automatically when consuming it.

Notes:
- Resumable: chunk names are deterministic, so re-running after an interruption reuses,
  resubmits, or resumes each chunk instead of duplicating work.
- AWS credentials are re-checked before every upload; a failure retries instead of crashing.
- Failed scenes are logged and written to `FAILED_SCENES_FILE` as they happen.
- Safe to interrupt (`KeyboardInterrupt`) and re-run.

In [ ]:
# Drop scenes that already have a tracking record in S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER
# -- i.e. scenes that were already processed to SLC and uploaded in a previous run.
paginator = s3_client.get_paginator("list_objects_v2")
processed_l0_scenes = set()
for page in paginator.paginate(
    Bucket=S3_BUCKET, Prefix=S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER.rstrip("/") + "/"
):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".json"):
            processed_l0_scenes.add(os.path.splitext(os.path.basename(obj["Key"]))[0])

before_count = len(L0_PRODUCTS)
L0_PRODUCTS = [
    scene
    for scene in L0_PRODUCTS
    if os.path.splitext(os.path.basename(scene))[0] not in processed_l0_scenes
]

print(
    f"Total of {len(processed_l0_scenes)} scene(s) already processed, found under "
    f"s3://{S3_BUCKET}/{S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER}."
)
print(f"Dropping already-processed scenes: {before_count} -> {len(L0_PRODUCTS)} scene(s) remaining.")

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

scene_chunks = chunk_scenes(L0_PRODUCTS, BATCH_CHUNK_SIZE)
print(
    f"Processing {len(L0_PRODUCTS)} scenes in {len(scene_chunks)} batch(es) of up to "
    f"{BATCH_CHUNK_SIZE} (CDSE allows only one active batch order at a time, so these run "
    "strictly sequentially)."
)

all_results: dict[str, str] = {}
failed_scenes: list[str] = []

try:
    for i, chunk in enumerate(scene_chunks):
        chunk_results = process_chunk(i, chunk)
        all_results.update(chunk_results)
        failed_scenes.extend(
            scene for scene, result in chunk_results.items() if result == "failed"
        )
except KeyboardInterrupt:
    print(
        "\nInterrupted. Re-run this cell to resume -- already-completed batches and "
        "uploaded scenes will not be resubmitted or re-uploaded."
    )
    raise

print(f"\nAll {len(scene_chunks)} batch(es) processed.")
overall_summary = {
    s: sum(1 for v in all_results.values() if v == s)
    for s in ("uploaded", "already_uploaded", "skipped", "failed")
}
print(f"Overall: {overall_summary}")

if failed_scenes:
    print(
        f"\n{len(failed_scenes)} scene(s) failed and were skipped -- also recorded in "
        f"{FAILED_SCENES_FILE} (updated live during the run). Investigate/resubmit manually:"
    )
    for scene in failed_scenes:
        print(f"  {scene}")

## 4. Process Scenes with ISCE3_RTC (Optional)
Now the EW level-1 SLC data is available, it can be processed using the isce3_rtc workflow. Below shows an example docker command. The **scene-path** is the path to the file uploaded from this process:

```bash
docker run --env-file .env --platform linux/amd64 \
-v $PWD/data:/home/rtc_user/working \
sar-pipeline-isce3-rtc \
--scene S1A_EW_SLC__1SDH_20250101T153651_20250101T153757_057252_070AF9_1F2E \
--scene-path https://deant-data-public-dev.s3.ap-southeast-2.amazonaws.com/s1_ew_slc/data/S1A_EW_SLC__1SDH_20250101T153651_20250101T153757_057252_070AF9_1F2E.SAFE.zip \
--resolution 40 \
--skip-upload-to-s3 \
--make-existing-products \
--burst-id-list t130_253088_ew5

```